In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
import numpy as np

########################################
# SEED
########################################

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

########################################
# NETWORKS
########################################

class PolicyNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3,16)
        self.fc2 = nn.Linear(16,1)

    def forward(self,x):

        h = torch.relu(self.fc1(x))
        return self.fc2(h)


class ValueNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(2,32)
        self.fc2 = nn.Linear(32,1)

    def forward(self,x):

        h = torch.relu(self.fc1(x))
        return self.fc2(h)

########################################
# DATASET
########################################

def generate_instance(N):

    weights = torch.rand(N)
    W = 1.0

    return weights, W


def generate_dataset(N,num_instances):

    dataset=[]

    for _ in range(num_instances):

        dataset.append(generate_instance(N))

    return dataset

########################################
# STATE
########################################

def init_state(N):

    X = torch.zeros(N,N)

    for i in range(N):
        X[i,i] = 1

    return X


def free_capacities(X,weights,W):

    load = torch.matmul(weights,X)

    return W-load


def energy(X):

    used = torch.sum(X,dim=0)>0

    return torch.sum(used).item()

########################################
# TEMPERATURE
########################################

def temperature(t,steps,T0=1.0,TK=0.01):

    alpha = (TK/T0)**(1/steps)

    return T0*(alpha**t)

########################################
# ACTION SELECTION
########################################

def select_item(policy,X,weights,free_caps,T):

    N = X.shape[0]

    bins = torch.argmax(X,dim=1)

    c_bi = free_caps[bins]

    T_vec = torch.full((N,),T)

    features = torch.stack([weights,c_bi,T_vec],dim=1)

    logits = policy(features).squeeze()

    probs = torch.softmax(logits,dim=0)

    dist = torch.distributions.Categorical(probs)

    i = dist.sample()

    return i,dist.log_prob(i)


def select_bin(policy,i,X,weights,free_caps,T):

    N = X.shape[0]

    w_i = weights[i]

    old_bin = torch.argmax(X[i])

    weight_vec = torch.full((N,),w_i)

    T_vec = torch.full((N,),T)

    features = torch.stack([weight_vec,free_caps,T_vec],dim=1)

    logits = policy(features).squeeze()

    mask = (free_caps>=w_i)

    mask[old_bin]=False

    logits[~mask] = -1e9

    probs = torch.softmax(logits,dim=0)

    dist = torch.distributions.Categorical(probs)

    j = dist.sample()

    return j,dist.log_prob(j)

########################################
# MOVE
########################################

def apply_move(X,i,j):

    X_new = X.clone()

    old = torch.argmax(X_new[i])

    X_new[i,old]=0
    X_new[i,j]=1

    return X_new

########################################
# METROPOLIS
########################################

def metropolis(X,X_new,T):

    E_old = energy(X)
    E_new = energy(X_new)

    delta = E_new-E_old

    if delta<=0:
        return X_new

    prob = np.exp(-delta/T)

    if random.random()<prob:
        return X_new

    return X

########################################
# ROLLOUT
########################################

def rollout(X,weights,W,steps,item_policy,bin_policy):

    log_probs=[]
    rewards=[]
    states=[]

    for t in range(steps):

        T = temperature(t,steps)

        E_old = energy(X)

        caps = free_capacities(X,weights,W)

        i,logp_i = select_item(item_policy,X,weights,caps,T)

        j,logp_j = select_bin(bin_policy,i,X,weights,caps,T)

        X_candidate = apply_move(X,i,j)

        X = metropolis(X,X_candidate,T)

        E_new = energy(X)

        reward = E_old-E_new

        state = torch.tensor([[E_new,T]],dtype=torch.float32)

        log_probs.append(logp_i+logp_j)

        rewards.append(reward)

        states.append(state)

    return X,log_probs,rewards,states

########################################
# RETURNS
########################################

def compute_returns(rewards,gamma=0.99):

    returns=[]

    G=0

    for r in reversed(rewards):

        G = r + gamma*G

        returns.insert(0,G)

    return torch.tensor(returns,dtype=torch.float32)

########################################
# PPO UPDATE
########################################

def ppo_update(item_policy,bin_policy,value_net,optimizer,
               log_probs,rewards,states):

    returns = compute_returns(rewards)

    values = torch.cat([value_net(s) for s in states]).squeeze()

    advantages = returns-values.detach()

    advantages = (advantages-advantages.mean())/(advantages.std()+1e-8)

    log_probs = torch.stack(log_probs)

    policy_loss = -(log_probs*advantages).mean()

    value_loss = (returns-values).pow(2).mean()

    loss = policy_loss + 0.5*value_loss

    optimizer.zero_grad()

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        list(item_policy.parameters())+
        list(bin_policy.parameters())+
        list(value_net.parameters()),
        1.0
    )

    optimizer.step()

    return loss.item()

########################################
# TRAINING
########################################

def train(dataset,epochs,steps,
          item_policy,bin_policy,value_net,optimizer):

    best_bins = float("inf")

    for epoch in range(epochs):

        total_bins=0

        for weights,W in dataset:

            X = init_state(len(weights))

            X,log_probs,rewards,states = rollout(
                X,weights,W,steps,item_policy,bin_policy)

            loss = ppo_update(
                item_policy,
                bin_policy,
                value_net,
                optimizer,
                log_probs,
                rewards,
                states)

            bins = energy(X)

            total_bins += bins

            best_bins = min(best_bins,bins)

        if epoch%5==0:

            print(
            "epoch",epoch,
            "| avg bins",total_bins/len(dataset),
            "| best",best_bins,
            "| loss",loss)

########################################
# EXPERIMENT
########################################

def run_experiment(N):

    seeds=[0,1,2,3,4]

    for seed in seeds:

        set_seed(seed)

        dataset = generate_dataset(N,20)

        item_policy = PolicyNet()
        bin_policy = PolicyNet()

        value_net = ValueNet()

        optimizer = optim.Adam(
            list(item_policy.parameters())+
            list(bin_policy.parameters())+
            list(value_net.parameters()),
            lr=1e-3)

        train(dataset,
              epochs=40,
              steps=20,
              item_policy=item_policy,
              bin_policy=bin_policy,
              value_net=value_net,
              optimizer=optimizer)

########################################

run_experiment(50)
run_experiment(100)

epoch 0 | avg bins 39.75 | best 38 | loss 37.386688232421875
epoch 5 | avg bins 39.7 | best 37 | loss 2.529561758041382
epoch 10 | avg bins 39.8 | best 36 | loss 4.237825393676758
epoch 15 | avg bins 40.45 | best 36 | loss 2.273909568786621
epoch 20 | avg bins 39.25 | best 35 | loss 4.825201511383057
epoch 25 | avg bins 39.5 | best 35 | loss 4.0787811279296875
epoch 30 | avg bins 39.4 | best 34 | loss 1.5450137853622437
epoch 35 | avg bins 39.65 | best 34 | loss 4.131725311279297
epoch 0 | avg bins 39.95 | best 36 | loss 2.5709643363952637
epoch 5 | avg bins 40.5 | best 36 | loss 2.412947654724121
epoch 10 | avg bins 39.95 | best 35 | loss 3.997368097305298
epoch 15 | avg bins 39.25 | best 35 | loss 4.424255847930908
epoch 20 | avg bins 39.9 | best 35 | loss 2.7681264877319336
epoch 25 | avg bins 40.0 | best 35 | loss 2.5690910816192627
epoch 30 | avg bins 39.85 | best 35 | loss 2.765432357788086
epoch 35 | avg bins 40.1 | best 35 | loss 3.3978960514068604
epoch 0 | avg bins 39.3 | bes